# Phases 3–6: dry-run, probes, bulk load, verification (invest mailing)

Follow-along driver for `update_invest_mailing.py` — the script stays the single
source of truth; this notebook imports its functions and walks the run step by step.

Sections 1–3 are local-only. Section 4 authenticates and runs **read-only** SOQL.
Sections 5 (probes: TWO records — one A, one C) and 6 (bulk load, ~6.1k in 2–3
signature jobs) **write to production** and are each gated behind an explicit flag
you flip by hand. Section 8 archives the batch to history after a verified load.

Reminder what the batch does: population **A** copies the full billing address block
(street/city/postal/country-name) into an empty mailing block, **B** writes the
country name only, **C** converts an existing bare ISO-2 mailing country to its
name in place — C is the one population that **overwrites** an existing value,
which is why it gets its own probe.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root
sys.path.insert(0, str(Path.cwd()))         # this folder: update_invest_mailing

import pandas as pd

import update_invest_mailing as um
from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_ID = "2026-08-26_invest_mailing_backfill"

# Phase 2 contract: the frozen per-population counts from 06's final cell
# (mirror 2026-08-26; A includes the 119 country-suppressed address-only rows:
# 108 suspect billing codes + 11 empty codes).
EXPECTED = {"A": 4467, "B": 440, "C": 1230}

db = MySQLClient(load_mysql_config())
print("connected | batch:", BATCH_ID)

## 1. Load the staged batch (local, read-only)

Same row selection the script uses: update rows, not excluded, payload present
(country OR address), not yet processed. After a successful load the open count
drops to 0 — flip `AFTER_LOAD = True` for post-load reruns.

In [ ]:
AFTER_LOAD = False  # True once the bulk load ran: open rows are then expected to be 0

rows = db.fetch_all("""
    SELECT row_id, sf_account_id, email, source,
           address, city, postal_code, country, _mailing_prev_country
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s AND _operation = 'update' AND _excluded = 0
      AND _mailing_processed_at IS NULL
      AND sf_account_id IS NOT NULL
      AND (   (country IS NOT NULL AND country <> '')
           OR (address IS NOT NULL AND address <> ''))
""", (BATCH_ID,))
print(f"{len(rows):,} rows open")

from collections import Counter
by_source = Counter(r["source"] for r in rows)
for src, n in sorted(by_source.items()):
    print(f"  {src}: {n:,}")

if not AFTER_LOAD:
    assert all(v is not None for v in EXPECTED.values()), "fill EXPECTED from 06 first"
    for pop, exp in EXPECTED.items():
        got = by_source.get(f"invest_mailing_{pop}", 0)
        assert got == exp, f"pop {pop}: {got} open != contract {exp}"
    print("matches the phase 2 contract")

## 2. Build the payload and eyeball it (local)

`row_to_sf_record` is the exact mapper the load uses; empty fields are omitted
entirely. Expect 2–3 signature groups: A with the full block (postal-less A rows
split off on their own), B/C with `PersonMailingCountry` only. **Every country
value must be a full name — the loader aborts on any bare ISO-2 code.**

In [ ]:
records = [um.row_to_sf_record(r) for r in rows]
groups = um.group_by_signature(records)
print("signature groups (fields -> records):")
for sig, recs in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(f"  {list(sig)}: {len(recs):,}")

preview = pd.DataFrame(records[:5])
preview

## 3. Phase 3 — dry-run (local, no Salesforce contact)

Runs the script itself with `--dry-run`: one CSV per signature group under
`<repo-root>/local_data/dry_run_invest_mailing_*.csv`. Open them and check:
no empty cells anywhere, every `PersonMailingCountry` a full English name
("Austria", "Germany", …), A rows carry a complete consistent address.

In [ ]:
import subprocess

proc = subprocess.run(
    [sys.executable, str(Path.cwd() / "update_invest_mailing.py"), BATCH_ID, "--dry-run"],
    cwd=str(Path.cwd().parent), capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"dry-run exited {proc.returncode}")

## 4. Authenticate + live spot-check (prod, READ-ONLY)

Runs the loader's per-source skip logic over **every open row** (at ~6.1k that is
~16 SOQL chunks, a few seconds — no need to sample): A/B rows whose mailing block
got filled live, C rows whose mailing country no longer equals the staged original
code. A handful of would-be skips is normal drift; a large share means the mirror
is stale — re-refresh and re-stage. This is exactly what the load's skip report
will say, minus the writing.

In [ ]:
from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env

sf = SalesforceClientCC(load_salesforce_cc_config_from_env())
sf.authenticate()
print("authenticated")

# Full batch, not a sample: ~6.1k ids = ~16 SOQL chunks, a few seconds.
live = um.fetch_live_mailing(sf, [str(r["sf_account_id"]) for r in rows])
reasons = Counter()
would_skip = []
for r in rows:
    reason = um.skip_reason(r, live.get(str(r["sf_account_id"])))
    if reason:
        reasons[f"{r['source']}: {reason.split(':')[0]}"] += 1
        would_skip.append({"sf_account_id": r["sf_account_id"], "source": r["source"],
                           "reason": reason})
print(f"live check on ALL {len(rows):,} open rows: {len(would_skip)} would be skipped")
for k, n in reasons.most_common():
    print(f"  {k}: {n}")
if would_skip:
    pd.DataFrame(would_skip).to_csv(
        Path.cwd().parent / "local_data" / "invest_mailing_preload_drift.csv", index=False)
    print("details -> local_data/invest_mailing_preload_drift.csv")

## 5. Phase 4 — probe records (prod, WRITES TWO ACCOUNTS)

Two probes, one per write shape:
- **A probe**: empty mailing block gets the full billing copy — readback shows a
  complete, internally consistent address.
- **C probe**: an existing code becomes a name — the cell prints the live BEFORE
  value first, then patches, then the readback shows street/city untouched.

Put the agreed 18-char Ids into the two variables (each must be in the open batch),
then flip `RUN_PROBE = True`. Eyeball both in the UI, business sign-off **before**
section 6.

In [ ]:
RUN_PROBE = False        # <- flip by hand for the two probe updates
PROBE_ACCOUNT_ID_A = ""  # <- an invest_mailing_A account from the open batch
PROBE_ACCOUNT_ID_C = ""  # <- an invest_mailing_C account from the open batch

MAILING_FIELDS = ("PersonMailingStreet, PersonMailingCity, PersonMailingPostalCode, "
                  "PersonMailingCountry")

if RUN_PROBE:
    for probe_id, want_src in [(PROBE_ACCOUNT_ID_A, "invest_mailing_A"),
                               (PROBE_ACCOUNT_ID_C, "invest_mailing_C")]:
        candidates = [r for r in rows if str(r["sf_account_id"]) == probe_id]
        assert candidates, f"account {probe_id!r} is not in the open batch"
        probe_row = candidates[0]
        assert probe_row["source"] == want_src, f"{probe_id} is {probe_row['source']}, expected {want_src}"

        before = sf.query_all(
            f"SELECT Id, {MAILING_FIELDS}, BillingStreet, BillingCity, BillingCountryCode__c "
            f"FROM Account WHERE Id = '{probe_id}'")["records"][0]
        payload = {k: v for k, v in um.row_to_sf_record(probe_row).items() if k != "Id"}

        pop = want_src[-1]  # 'A' or 'C'
        print(f"===== probe {pop} | {probe_id} | {probe_row['email']} =====")
        print(f"{'field':<26} {'live BEFORE':<22} {'will become':<22}")
        for f in ["PersonMailingStreet", "PersonMailingCity",
                  "PersonMailingPostalCode", "PersonMailingCountry"]:
            old = before.get(f) or "(empty)"
            new = payload.get(f, "(not sent - unchanged)")
            print(f"{f:<26} {str(old):<22} {str(new):<22}")
        print(f"(billing source: {before.get('BillingStreet')}, {before.get('BillingCity')}, "
              f"{before.get('BillingCountryCode__c')})")

        sf.update_account_by_id(probe_id, payload)
        print(f"-> updated {probe_id}\n")
else:
    print("probes skipped (RUN_PROBE = False)")

In [ ]:
if RUN_PROBE:
    for label, probe_id in [("A", PROBE_ACCOUNT_ID_A), ("C", PROBE_ACCOUNT_ID_C)]:
        rec = sf.query_all(
            f"SELECT Id, PersonEmail, {MAILING_FIELDS}, "
            f"NationalityCountryCode__pc, LastModifiedDate "
            f"FROM Account WHERE Id = '{probe_id}'")["records"][0]
        print(f"===== AFTER probe {label} | {probe_id} | {rec.get('PersonEmail')} =====")
        for f in ["PersonMailingStreet", "PersonMailingCity",
                  "PersonMailingPostalCode", "PersonMailingCountry",
                  "NationalityCountryCode__pc", "LastModifiedDate"]:
            print(f"  {f:<28} {rec.get(f) or '(empty)'}")
        print()
    print("check A: full consistent address, country a full name")
    print("check C: only PersonMailingCountry changed (code -> name), street/city untouched")

## 6. Phase 5 — bulk load (prod, WRITES ~6.1k ACCOUNTS)

Runs the script itself, so the real run is exactly what was dry-run — plus the full
per-source live skip-check, the bare-code abort, the duplicate-Id abort, per-job
re-authentication, abort after 3 consecutive job errors, and the per-job
`_mailing_processed_at` writeback. A crashed run is resumable by re-running the same
command. Non-zero exit on ANY record failure or job error.
Expect 2–3 jobs (one per signature), a few minutes; monitor in Setup → Bulk Data Load Jobs.

**Does not run without Arsal's explicit go-ahead.** Flip `RUN_LOAD = True` by hand.

In [ ]:
import subprocess

RUN_LOAD = False  # <- Arsal's explicit go-ahead required (phase 5 gate)

if RUN_LOAD:
    assert all(v is not None for v in EXPECTED.values()), (
        "fill EXPECTED in the setup cell with the frozen counts from 06 "
        "before loading - the contract check must not be skipped for the real run"
    )
    proc = subprocess.run(
        [sys.executable, str(Path.cwd() / "update_invest_mailing.py"), BATCH_ID],
        cwd=str(Path.cwd().parent), capture_output=True, text=True,
    )
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"load exited {proc.returncode} - fix before re-running")
else:
    print("bulk load skipped (RUN_LOAD = False)")

## 7. Phase 6 — verification (prod, read-only)

Live SOQL: `PersonMailingCountry` coverage across invest accounts should be near
total (everything but population D's partial blocks, the unreachable rest, and
live skips), and **zero** values should remain bare ISO-2 codes — SOQL has no
regex, so the value list is pulled and checked in Python. Plus staging writeback
completeness: `still_open` must be 0.

In [ ]:
import re

BASE = "FROM Account WHERE IsPersonAccount = true AND InvestCustomer__pc = true"
total = sf.query_all(f"SELECT COUNT(Id) n {BASE}")["records"][0]["n"]
recs = sf.query_all(
    f"SELECT PersonMailingCountry v, COUNT(Id) n {BASE} "
    f"AND PersonMailingCountry != null GROUP BY PersonMailingCountry ORDER BY COUNT(Id) DESC"
)["records"]
filled = sum(r["n"] for r in recs)
bare = [(r["v"], r["n"]) for r in recs if re.fullmatch(r"[A-Za-z]{2}", (r["v"] or "").strip())]
print(f"invest accounts: {total:,} | PersonMailingCountry filled: {filled:,} ({filled/total*100:.1f}%)")
print("values:")
for r in recs:
    print(f"  {r['v']}: {r['n']:,}")
if bare:
    print(f"\nWARNING - still bare ISO-2 codes: {bare}")
else:
    print("\nno bare ISO-2 codes left")

open_rows = db.fetch_df("""
    SELECT SUM(_mailing_processed_at IS NULL AND _excluded = 0) AS still_open,
           SUM(_excluded = 1) AS excluded,
           SUM(_mailing_processed_at IS NOT NULL) AS processed,
           COUNT(*) AS total
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
""", (BATCH_ID,))
print()
print(open_rows.to_string(index=False))
print("\nmismatch? check <repo-root>/local_data/skipped_invest_mailing_* and failed_invest_mailing_*")

## 8. Archive the batch (local MySQL, after verified load only)

Snapshots the batch into `crm_imp_person_accounts_history` and deletes it from the
staging table. Only run once section 7 is clean: `still_open = 0` and the live
values check out. (Raw connector: the proc returns a result set `MySQLClient`
can't consume; the proc commits internally.)

**The archive proc's column list does NOT include `_mailing_prev_country`** (or
the bookkeeping columns) — the history table has no such column — so the
original ISO-2 code of every population-C account would be lost forever.
The cell therefore exports the full revert/audit record to `local_data/`
**before** calling the proc, and refuses to archive if the export fails.

In [ ]:
RUN_ARCHIVE = False  # <- flip by hand after section 7 is verified clean

if RUN_ARCHIVE:
    # The proc drops _mailing_prev_country (not in the history column list):
    # persist the before/after record first - without it, "revert the C
    # conversion for account X" is unanswerable after archiving.
    snapshot = db.fetch_df("""
        SELECT row_id, sf_account_id, source, country, _mailing_prev_country,
               _mailing_processed_at, _excluded, _exclude_reason
        FROM crm_imp_person_accounts
        WHERE _batch_id = %s
    """, (BATCH_ID,))
    snap_out = Path.cwd().parent / "local_data" / f"archive_snapshot_{BATCH_ID}.csv"
    snapshot.to_csv(snap_out, index=False)
    assert len(snapshot) > 0 and snap_out.exists(), "snapshot export failed - NOT archiving"
    print(f"pre-archive snapshot: {len(snapshot):,} rows -> {snap_out}")

    import mysql.connector
    cfg = load_mysql_config()
    conn = mysql.connector.connect(host=cfg.host, user=cfg.user, password=cfg.password,
                                   database=cfg.database, port=cfg.port)
    cur = conn.cursor(dictionary=True)
    cur.execute("CALL sp_archive_crm_imp_person_accounts(%s, %s)",
                (BATCH_ID, "update_invest_mailing.py"))
    while True:
        if cur.with_rows:
            for r in cur.fetchall():
                print(r)
        if cur.nextset() is None:
            break
    cur.close()
    conn.close()

    left = db.fetch_one(
        "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s",
        (BATCH_ID,),
    )["n"]
    hist = db.fetch_one(
        "SELECT COUNT(*) AS n FROM crm_imp_person_accounts_history WHERE _batch_id = %s",
        (BATCH_ID,),
    )["n"]
    print(f"staging rows left: {left} | history rows: {hist}")
    assert left == 0, "archive incomplete - staging rows remain"
else:
    print("archive skipped (RUN_ARCHIVE = False)")